# Sticky Note Blindness

Solution author: Cowile

CLIP's ViT reads the sticky note text and predicts the wrong class. Text is directional, so if you flip an image horizontally, the object stays the same but the text becomes unreadable to CLIP, and the typographic attack disappears.

<img src="https://imgur.com/rMlP2wp.png" width="750">

## 0. Setup

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA = Path('/kaggle/input/sticky-note-blindness-aicc-round-4')
BS = 64
print(f'Device: {DEVICE}')

model = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch16')
print(f'CLIP ViT-B/16 on {DEVICE}')

Device: cuda


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP ViT-B/16 on cuda


In [2]:
text_embs = torch.load(DATA / 'class_embeddings.pt', weights_only=True).to(DEVICE).float()
text_embs = F.normalize(text_embs, dim=-1)

metadata = pd.read_csv(DATA / 'metadata.csv')
valid_ids = set(metadata['sample_id'].tolist())
all_paths = sorted((DATA / 'images' / 'images').glob('*.png'), key=lambda p: int(p.stem))
image_paths = [p for p in all_paths if int(p.stem) in valid_ids]
images = [Image.open(p).convert('RGB') for p in tqdm(image_paths, desc='Loading')]

N = len(images)
K = text_embs.shape[0]
print(f'{N} images  |  {K} classes')

Loading: 100%|██████████| 348/348 [00:05<00:00, 67.94it/s]

348 images  |  764 classes


In [3]:
@torch.inference_mode()
def encode(imgs, batch_size=BS):
    embs = []
    for i in range(0, len(imgs), batch_size):
        px = processor(images=imgs[i:i+batch_size], return_tensors='pt').to(DEVICE)
        vision_out = model.vision_model(pixel_values=px['pixel_values'])
        pooled = vision_out.pooler_output
        projected = model.visual_projection(pooled)
        embs.append(F.normalize(projected.float(), dim=-1).cpu())
    return torch.cat(embs)

def score(embs):
    return embs @ text_embs.cpu().T

## Solution 1: Flip Only (~87%)

The simplest defense. Mirror the image horizontally so text becomes unreadable, then classify the flipped version. This works because objects are approximately bilaterally symmetric: a dog looks like a dog mirrored. But text reads left-to-right, and flipping destroys its semantic content.

In [ ]:
images_flip = [img.transpose(Image.FLIP_LEFT_RIGHT) for img in images]

emb_orig = encode(images)
emb_flip = encode(images_flip)

s_orig = score(emb_orig)
s_flip = score(emb_flip)

## Solution 2: MirrorCLIP (~89%)

### The Insight

CLIP's 512-dimensional embedding space is partially separable: some dimensions respond to visual features (shape, color, texture) and others to textual features (characters on the sticky note). We can tell them apart without any labels by checking whether each dimension agrees across the original and flipped image:

```
agreement[d] = emb_orig[d] * emb_flip[d]
```


If `agreement[d] > 0`, dimension `d` kept the same sign after flipping, so it tracks a visual feature. If `agreement[d] < 0`, the sign flipped, meaning it was encoding textual feature. For visual dimensions we reinforce the signal by summing both views; for textual dimensions we discard the corrupted original and keep only the flip.

Inspired by Wang et al., [MirrorCLIP: Disentangling text from visual images through reflection](https://openreview.net/pdf?id=FYm8coxdiR).


In [ ]:
def mirrorclip(emb_orig, emb_flip):
    # dims where orig and flip agree -> visual (keep both)
    # dims where they disagree -> textual (keep only the flip)
    
    visual = (emb_orig * emb_flip) > 0
    out = torch.where(visual, emb_orig + emb_flip, emb_flip)
    return F.normalize(out, dim=-1)

emb_clean = mirrorclip(emb_orig, emb_flip)
preds_mirrorclip = score(emb_clean).argmax(1).tolist()

We can inspect the agreement mask to understand what is happening per image. On average, roughly 106 out of 512 dimensions are textual (sign-disagreeing), meaning about 20% of CLIP's embedding space is hijacked by text.

In [6]:
agreement = emb_orig * emb_flip
n_textual = (agreement < 0).float().sum(dim=1)
n_visual = (agreement > 0).float().sum(dim=1)

print('Per-image statistics:')
print(f'  Visual dims:  {n_visual.mean():.0f} avg  (range {n_visual.min():.0f}-{n_visual.max():.0f})')
print(f'  Textual dims: {n_textual.mean():.0f} avg  (range {n_textual.min():.0f}-{n_textual.max():.0f})')
print(f'  Textual fraction: {(n_textual / 512).mean():.1%}')

Per-image statistics:
  Visual dims:  406 avg  (range 351-486)
  Textual dims: 106 avg  (range 26-161)
  Textual fraction: 20.6%


## Submission

In [7]:
sample_ids = [int(p.stem) for p in image_paths]
submission = pd.DataFrame({'sample_id': sample_ids, 'label': preds_mirrorclip})
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv ({len(submission)} rows)')
print(submission.head())

Saved submission.csv (348 rows)
   sample_id  label
0          0    237
1          1    355
2          2     13
3          3    693
4          4    164


to push the score further, we can blend in additional text-destroying views (e.g. center crop) at the logit level, and mix back a fraction of the original embedding to recover visual information lost during disentanglement.